# 01 — Exploratory Data Analysis (EDA)
**Dataset:** Credit Card Fraud Detection — Kaggle (284,807 transaksione)

Qëllimi: Kuptojmë strukturën e dataset-it, shpërndarjen e variablave, çekuilibrin e klasave dhe korrelacionet mes featureve.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print(f'Dataset u ngarkua me sukses!')
print(f'Dimensionet: {df.shape[0]:,} rreshta  x  {df.shape[1]} kolona')
df.head()

## 1. Informacioni Bazë i Dataset-it
Kontrollojmë dimensionet, tipet e kolonave, vlerat null dhe duplikatet.

In [ ]:
print("=" * 50)
print("INFORMACIONI BAZË I DATASET-IT")
print("=" * 50)
print(f"\nNumri i rreshtave : {df.shape[0]:,}")
print(f"Numri i kolonave  : {df.shape[1]}")
print(f"\nTipet e kolonave:")
print(df.dtypes.value_counts().to_string())
print(f"\nVlera null (totale): {df.isnull().sum().sum()}")
print(f"Rreshta duplikatë : {df.duplicated().sum()}")

In [ ]:
df.describe()

## 2. Analiza e Çekuilibrit të Klasave (Class Imbalance)
Njëra nga sfidat kryesore në fraud detection: klasa e mashtrimit është shumë e nënpërfaqësuar.

In [ ]:
class_counts = df['Class'].value_counts()
class_pct    = df['Class'].value_counts(normalize=True) * 100

print("=" * 50)
print("SHPËRNDARJA E KLASAVE")
print("=" * 50)
print(f"Transaksione legjitime  (0): {class_counts[0]:>7,}  ({class_pct[0]:.3f}%)")
print(f"Transaksione mashtruese (1): {class_counts[1]:>7,}  ({class_pct[1]:.3f}%)")
print(f"\nRaporti i çekuilibrit: {class_counts[0] / class_counts[1]:.0f}:1")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Legjitime', 'Mashtruese'], class_counts.values,
            color=['steelblue', 'tomato'], edgecolor='black', linewidth=0.5)
axes[0].set_title('Numri i Transaksioneve sipas Klasës', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold', fontsize=11)

axes[1].pie(class_counts.values,
            labels=['Legjitime (99.83%)', 'Mashtruese (0.17%)'],
            colors=['steelblue', 'tomato'],
            autopct='%1.3f%%', startangle=90, explode=(0, 0.08))
axes[1].set_title('Përqindja e Klasave', fontweight='bold')

plt.suptitle('Class Imbalance — Credit Card Fraud Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Analiza e Featureve Amount dhe Time
`Amount` (shuma e transaksionit) dhe `Time` (sekonda nga transaksioni i parë) janë dy featurat e patransformuara me PCA.

In [ ]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print("=== Statistikat e Amount sipas Klasës ===")
print(df.groupby('Class')['Amount'].describe().round(2))
print(f"\nAmount mesatar — Legjitime : ${legit['Amount'].mean():.2f}")
print(f"Amount mesatar — Mashtruese: ${fraud['Amount'].mean():.2f}")
print(f"Amount maksimal — Legjitime : ${legit['Amount'].max():,.2f}")
print(f"Amount maksimal — Mashtruese: ${fraud['Amount'].max():,.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['Amount'], bins=100, color='steelblue', alpha=0.8, edgecolor='white')
axes[0, 0].set_title('Shpërndarja e Amount (të gjitha)')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Frekuenca')
axes[0, 0].set_yscale('log')

axes[0, 1].boxplot([legit['Amount'], fraud['Amount']],
                   labels=['Legjitime', 'Mashtruese'],
                   patch_artist=True,
                   boxprops=dict(facecolor='lightblue'))
axes[0, 1].set_title('Amount sipas Klasës (Boxplot)')
axes[0, 1].set_ylabel('Amount ($)')

axes[1, 0].hist(df['Time'] / 3600, bins=100, color='seagreen', alpha=0.8, edgecolor='white')
axes[1, 0].set_title('Shpërndarja e Time (të gjitha)')
axes[1, 0].set_xlabel('Koha (orë)')
axes[1, 0].set_ylabel('Frekuenca')

axes[1, 1].hist(legit['Time'] / 3600, bins=80, alpha=0.6, label='Legjitime', color='steelblue')
axes[1, 1].hist(fraud['Time'] / 3600, bins=80, alpha=0.6, label='Mashtruese', color='tomato')
axes[1, 1].set_title('Time sipas Klasës')
axes[1, 1].set_xlabel('Koha (orë)')
axes[1, 1].legend()

plt.suptitle('Analiza e Featureve Amount dhe Time', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/amount_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analiza e Korrelacioneve
Gjejmë cilat feature V1–V28 kanë korrelacionin më të lartë me variablin target `Class`.

In [ ]:
corr_with_class = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)

print("=== Top 10 Features me Korrelacion ndaj Class ===")
print(corr_with_class.head(10).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

v_cols = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time', 'Class']
corr_matrix = df[v_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            ax=axes[0], cbar_kws={'shrink': 0.8}, linewidths=0.1)
axes[0].set_title('Matrica e Korrelacionit', fontweight='bold')

colors = ['tomato' if x > 0 else 'steelblue' for x in corr_with_class.head(15).values]
axes[1].barh(corr_with_class.head(15).index[::-1],
             corr_with_class.head(15).values[::-1], color=colors[::-1])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 15 Features: Korrelacioni me Class', fontweight='bold')
axes[1].set_xlabel('Koeficienti i korrelacionit')

plt.suptitle('Analiza e Korrelacioneve', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Krahasimi i Featureve: Fraud vs Legjitime
Shikojmë shpërndarjen e 6 featureve me korrelacionin më të lartë ndaj Class.

In [ ]:
top_features = corr_with_class.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    axes[i].hist(legit[feature], bins=60, alpha=0.6, label='Legjitime',
                 color='steelblue', density=True)
    axes[i].hist(fraud[feature], bins=60, alpha=0.6, label='Mashtruese',
                 color='tomato', density=True)
    axes[i].set_title(f'Shpërndarja e {feature}', fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Densiteti')
    axes[i].legend()

plt.suptitle('Top 6 Features: Fraud vs Legjitime', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()